In [1]:
import asyncio, time

async def fake_llm(name: str, seconds: float) -> str:
    print(f"{name}: yuborildi")
    await asyncio.sleep(seconds)
    print(f"{name}: javob keldi")
    return f"{name} natijasi"

In [5]:
result = fake_llm("A", 3)

print(result)
print(await result)

/var/folders/hc/4rplr_mn483_lyfqfy2m95tw0000gn/T/ipykernel_92239/3670886300.py:1: RuntimeWarning: coroutine 'fake_llm' was never awaited
  result = fake_llm("A", 3)


<coroutine object fake_llm at 0x109f53a00>
A: yuborildi
A: javob keldi
A natijasi


In [6]:
t0 = time.perf_counter()
a = await fake_llm("A", 1)     # 1 s kutadi
b = await fake_llm("B", 1)     # keyin yana 1 s
c = await fake_llm("C", 1)     # keyin yana 1 s
print(f"{time.perf_counter() - t0:.1f} s")  

A: yuborildi
A: javob keldi
B: yuborildi
B: javob keldi
C: yuborildi
C: javob keldi
3.0 s


In [8]:
t0 = time.perf_counter()
a,b,c = await asyncio.gather(
    fake_llm("A", 1),
    fake_llm("B", 1),
    fake_llm("C", 1),
)
print(f"{time.perf_counter() - t0:.1f} s")  

A: yuborildi
B: yuborildi
C: yuborildi
A: javob keldi
B: javob keldi
C: javob keldi
1.0 s


In [9]:
start = time.perf_counter()
questions = ["savol 1", "savol 2", "savol 3", "savol 4", "savol 5"]

res = await asyncio.gather(*[fake_llm(question, 1) for question in questions])

print(res)

savol 1: yuborildi
savol 2: yuborildi
savol 3: yuborildi
savol 4: yuborildi
savol 5: yuborildi
savol 1: javob keldi
savol 2: javob keldi
savol 3: javob keldi
savol 4: javob keldi
savol 5: javob keldi
['savol 1 natijasi', 'savol 2 natijasi', 'savol 3 natijasi', 'savol 4 natijasi', 'savol 5 natijasi']


In [10]:
task = asyncio.create_task(fake_llm("Xulosa", 3))
print("Another task...")
await asyncio.sleep(1)
print("Still another task")

result = await task

print(result)

Another task...
Xulosa: yuborildi
Still another task
Xulosa: javob keldi
Xulosa natijasi


In [14]:
coros = [fake_llm("A", 3), fake_llm("B", 1), fake_llm("C", 2)]

for next_done in asyncio.as_completed(coros):
    result = await next_done
    print("Tayyor: ", result)

C: yuborildi
A: yuborildi
B: yuborildi
B: javob keldi
Tayyor:  B natijasi
C: javob keldi
Tayyor:  C natijasi
A: javob keldi
Tayyor:  A natijasi


In [16]:
sem = asyncio.Semaphore(5)
async def limited_llm(name: str) -> str:
    async with sem:
        return await fake_llm(name, 1)

t0 = time.perf_counter()
results = await asyncio.gather(*[limited_llm(f"#{i}") for i in range(9)])
print(f"{time.perf_counter() - t0:.1f} s")

#0: yuborildi
#1: yuborildi
#2: yuborildi
#3: yuborildi
#4: yuborildi
#0: javob keldi
#1: javob keldi
#2: javob keldi
#3: javob keldi
#4: javob keldi
#5: yuborildi
#6: yuborildi
#7: yuborildi
#8: yuborildi
#5: javob keldi
#6: javob keldi
#7: javob keldi
#8: javob keldi
2.0 s
